# Phase 2: CAPM and Single-Index Factor Research

## Project context

This notebook extends the market data foundation from Phase 1 into benchmark-relative research. It estimates each asset's sensitivity to SPY using CAPM-style metrics and a single-index regression framework.

## CAPM objective

The goal is to understand how each asset behaves relative to the SPY benchmark. This phase focuses on beta, alpha, benchmark correlation, tracking error, single-index regression outputs, and rolling beta stability. It does not perform portfolio optimization or backtesting.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.config import OUTPUTS_DIR, FIGURES_DIR
from src.factors import (
    load_returns,
    split_assets_and_benchmark,
    calculate_capm_metrics,
    run_single_index_regression,
    calculate_rolling_beta,
    summarize_factor_results,
)
from src.visualization import plot_factor_bar, plot_rolling_beta

BENCHMARK = "SPY"
RETURNS_PATH = OUTPUTS_DIR / "returns" / "daily_returns.csv"
FACTORS_DIR = OUTPUTS_DIR / "factors"
FACTORS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
def _normalize(values):
    values = np.asarray(values, dtype=float)
    finite = np.isfinite(values)
    if not finite.any():
        return np.zeros_like(values)
    vmin = np.nanmin(values[finite])
    vmax = np.nanmax(values[finite])
    if np.isclose(vmin, vmax):
        return np.full_like(values, 0.5, dtype=float)
    return (values - vmin) / (vmax - vmin)


def save_plotly_or_pillow(fig, output_path, chart_type, data):
    output_path = Path(output_path)
    try:
        fig.write_image(str(output_path), width=1200, height=700, scale=2)
        return "plotly"
    except Exception as exc:
        draw_basic_png(output_path, chart_type, data, str(exc))
        return "pillow_fallback"


def draw_basic_png(output_path, chart_type, data, reason):
    width, height = 1200, 700
    margin = 80
    image = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default()
    draw.text((margin, 24), output_path.stem.replace("_", " ").title(), fill="black", font=font)
    draw.text((margin, 48), "Rendered with Pillow fallback because Plotly PNG export was unavailable.", fill="#555555", font=font)
    left, top, right, bottom = margin, 110, width - margin, height - margin
    draw.rectangle((left, top, right, bottom), outline="#333333")

    if chart_type == "bar":
        frame = data.copy()
        values = frame["value"].astype(float).values
        labels = frame["ticker"].astype(str).tolist()
        zero = 0 if np.nanmin(values) < 0 < np.nanmax(values) else np.nanmin(values)
        scale_values = _normalize(np.append(values, zero))[:-1]
        zero_y = bottom - (bottom - top) * _normalize([zero, np.nanmin(values), np.nanmax(values)])[0]
        bar_width = (right - left) / max(len(values), 1) * 0.65
        for i, value in enumerate(values):
            x = left + (right - left) * (i + 0.5) / len(values)
            y = bottom - (bottom - top) * scale_values[i]
            draw.rectangle((x - bar_width / 2, min(y, zero_y), x + bar_width / 2, max(y, zero_y)), fill="#1f77b4")
            draw.text((x - 14, bottom + 10), labels[i], fill="black", font=font)
            draw.text((x - 18, min(y, zero_y) - 16), f"{value:.2f}", fill="black", font=font)
    elif chart_type == "lines":
        frame = data.dropna(how="all")
        if len(frame) > 260:
            frame = frame.iloc[np.linspace(0, len(frame) - 1, 260).astype(int)]
        colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f"]
        for idx, col in enumerate(frame.columns):
            values = frame[col].astype(float).values
            y_norm = _normalize(values)
            points = []
            for i, value in enumerate(y_norm):
                x = left + (right - left) * i / max(len(y_norm) - 1, 1)
                y = bottom - (bottom - top) * value
                points.append((x, y))
            if len(points) > 1:
                draw.line(points, fill=colors[idx % len(colors)], width=2)
            draw.text((right - 130, top + 18 * idx), str(col), fill=colors[idx % len(colors)], font=font)
    image.save(output_path)


## Load daily returns

In [3]:
returns = load_returns(RETURNS_PATH)
display(returns.head())
print(f"Daily returns shape: {returns.shape}")
print(f"Date range: {returns.index.min().date()} to {returns.index.max().date()}")

,AAPL,MSFT,JPM,PG,XOM,JNJ,KO,NVDA,SPY
date,,,,,,,,,
2019-01-03,-0.099608,-0.036788,-0.014212,-0.007011,-0.015354,-0.015890,-0.006179,-0.060417,-0.023863
2019-01-04,0.042690,0.046509,0.036865,0.020410,0.036870,0.016783,0.019940,0.064068,0.033496
2019-01-07,-0.002226,0.001275,0.000695,-0.004000,0.005201,-0.006415,-0.013033,0.052941,0.007885
2019-01-08,0.019063,0.007251,-0.001886,0.003691,0.007271,0.023227,0.011289,-0.024895,0.009396
2019-01-09,0.016981,0.014299,-0.001690,-0.016331,0.005275,-0.007926,-0.019166,0.019667,0.004673


Daily returns shape: (1838, 9)
Date range: 2019-01-03 to 2026-04-27


## Benchmark and asset split

In [4]:
asset_returns, benchmark_returns = split_assets_and_benchmark(returns, benchmark=BENCHMARK)
print(f"Benchmark: {BENCHMARK}")
print(f"Assets: {', '.join(asset_returns.columns)}")
print(f"Aligned asset return shape: {asset_returns.shape}")

Benchmark: SPY
Assets: AAPL, MSFT, JPM, PG, XOM, JNJ, KO, NVDA
Aligned asset return shape: (1838, 8)


## CAPM beta analysis

In [5]:
capm_metrics = calculate_capm_metrics(asset_returns, benchmark_returns, risk_free_rate=0.0)
capm_metrics.to_csv(FACTORS_DIR / "capm_metrics.csv", index_label="ticker")
display(capm_metrics.style.format({
    "beta": "{:.2f}",
    "alpha": "{:.2%}",
    "annualized_return": "{:.2%}",
    "annualized_volatility": "{:.2%}",
    "correlation_to_benchmark": "{:.2f}",
    "r_squared": "{:.2f}",
    "expected_return_capm": "{:.2%}",
    "tracking_error": "{:.2%}",
}))


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/rovs/Library/Python/3.11/lib/python/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/rovs/Library/Python/3.11/lib/python/site-packages/ipykernel/kernelapp.py", line 736, in start
    se

AttributeError: _ARRAY_API not found

,beta,alpha,annualized_return,annualized_volatility,correlation_to_benchmark,r_squared,expected_return_capm,tracking_error
ticker,,,,,,,,
NVDA,1.82,44.74%,76.10%,50.96%,0.70,0.49,31.36%,39.77%
AAPL,1.22,10.00%,30.92%,30.84%,0.77,0.60,20.92%,20.04%
MSFT,1.15,2.95%,22.74%,28.60%,0.79,0.62,19.79%,17.86%
JPM,1.07,1.88%,20.29%,29.69%,0.71,0.50,18.41%,21.08%
XOM,0.77,3.00%,16.31%,31.08%,0.49,0.24,13.31%,27.49%
KO,0.53,1.14%,10.19%,19.73%,0.52,0.27,9.05%,19.22%
PG,0.49,1.40%,9.78%,20.12%,0.47,0.22,8.37%,20.36%
JNJ,0.42,4.01%,11.22%,19.12%,0.43,0.18,7.21%,20.67%


## Single-index regression

In [6]:
single_index_regression = run_single_index_regression(asset_returns, benchmark_returns)
single_index_regression.to_csv(FACTORS_DIR / "single_index_regression.csv", index_label="ticker")
regression_methods = single_index_regression["method"].value_counts().to_dict()
display(single_index_regression.style.format({
    "alpha_daily": "{:.4%}",
    "alpha_annualized_simple": "{:.2%}",
    "beta": "{:.2f}",
    "alpha_pvalue": "{:.4f}",
    "beta_pvalue": "{:.4f}",
    "r_squared": "{:.2f}",
    "residual_volatility": "{:.2%}",
}))
regression_methods

AttributeError: _ARRAY_API not found

,alpha_daily,alpha_annualized_simple,beta,alpha_pvalue,beta_pvalue,r_squared,residual_volatility,method
ticker,,,,,,,,
NVDA,0.1472%,37.10%,1.82,nan,nan,nan,nan%,numpy_polyfit_fallback
AAPL,0.0399%,10.05%,1.22,nan,nan,nan,nan%,numpy_polyfit_fallback
MSFT,0.0163%,4.11%,1.15,nan,nan,nan,nan%,numpy_polyfit_fallback
JPM,0.0152%,3.82%,1.07,nan,nan,nan,nan%,numpy_polyfit_fallback
XOM,0.0245%,6.17%,0.77,nan,nan,nan,nan%,numpy_polyfit_fallback
KO,0.0091%,2.30%,0.53,nan,nan,nan,nan%,numpy_polyfit_fallback
PG,0.0107%,2.69%,0.49,nan,nan,nan,nan%,numpy_polyfit_fallback
JNJ,0.0198%,5.00%,0.42,nan,nan,nan,nan%,numpy_polyfit_fallback


{'numpy_polyfit_fallback': 8}

## Rolling beta analysis

In [7]:
rolling_beta = calculate_rolling_beta(asset_returns, benchmark_returns, window=126)
rolling_beta.to_csv(FACTORS_DIR / "rolling_beta.csv", index_label="date")
display(rolling_beta.tail())
print(f"Rolling beta shape: {rolling_beta.shape}")

,AAPL,MSFT,JPM,PG,XOM,JNJ,KO,NVDA
date,,,,,,,,
2026-04-21,0.875378,0.949489,1.026236,0.025023,-0.429712,0.021854,-0.033948,1.832574
2026-04-22,0.860629,0.965255,1.009155,0.024942,-0.426176,0.018773,-0.034946,1.851227
2026-04-23,0.858633,0.982188,1.008268,0.014827,-0.427557,0.008921,-0.042954,1.854184
2026-04-24,0.838737,1.002926,0.990187,0.037004,-0.427430,-0.000693,-0.043963,1.880088
2026-04-27,0.840658,1.006077,0.991609,0.036574,-0.435054,0.002237,-0.035516,1.879825


Rolling beta shape: (1713, 8)


## Factor summary output

In [8]:
factor_summary = summarize_factor_results(capm_metrics, single_index_regression)
factor_summary.to_csv(FACTORS_DIR / "factor_summary.csv", index_label="ticker")
display(factor_summary.style.format({
    "beta": "{:.2f}",
    "alpha": "{:.2%}",
    "annualized_return": "{:.2%}",
    "annualized_volatility": "{:.2%}",
    "correlation_to_benchmark": "{:.2f}",
    "r_squared": "{:.2f}",
    "expected_return_capm": "{:.2%}",
    "tracking_error": "{:.2%}",
    "alpha_daily": "{:.4%}",
    "alpha_annualized_simple": "{:.2%}",
    "regression_beta": "{:.2f}",
}))

,beta,alpha,annualized_return,annualized_volatility,correlation_to_benchmark,r_squared,expected_return_capm,tracking_error,alpha_daily,alpha_annualized_simple,regression_beta,alpha_pvalue,beta_pvalue,regression_r_squared,residual_volatility,method
ticker,,,,,,,,,,,,,,,,
NVDA,1.82,44.74%,76.10%,50.96%,0.70,0.49,31.36%,39.77%,0.1472%,37.10%,1.82,nan,nan,nan,nan,numpy_polyfit_fallback
AAPL,1.22,10.00%,30.92%,30.84%,0.77,0.60,20.92%,20.04%,0.0399%,10.05%,1.22,nan,nan,nan,nan,numpy_polyfit_fallback
MSFT,1.15,2.95%,22.74%,28.60%,0.79,0.62,19.79%,17.86%,0.0163%,4.11%,1.15,nan,nan,nan,nan,numpy_polyfit_fallback
JPM,1.07,1.88%,20.29%,29.69%,0.71,0.50,18.41%,21.08%,0.0152%,3.82%,1.07,nan,nan,nan,nan,numpy_polyfit_fallback
XOM,0.77,3.00%,16.31%,31.08%,0.49,0.24,13.31%,27.49%,0.0245%,6.17%,0.77,nan,nan,nan,nan,numpy_polyfit_fallback
KO,0.53,1.14%,10.19%,19.73%,0.52,0.27,9.05%,19.22%,0.0091%,2.30%,0.53,nan,nan,nan,nan,numpy_polyfit_fallback
PG,0.49,1.40%,9.78%,20.12%,0.47,0.22,8.37%,20.36%,0.0107%,2.69%,0.49,nan,nan,nan,nan,numpy_polyfit_fallback
JNJ,0.42,4.01%,11.22%,19.12%,0.43,0.18,7.21%,20.67%,0.0198%,5.00%,0.42,nan,nan,nan,nan,numpy_polyfit_fallback


## Figures

In [9]:
selected_rolling_assets = ["NVDA", "AAPL", "JPM", "PG"]

figure_specs = {
    "capm_beta_by_asset.png": (
        plot_factor_bar(capm_metrics, "beta", "CAPM Beta by Asset", "Beta"),
        "bar",
        capm_metrics[["beta"]].rename(columns={"beta": "value"}).reset_index().rename(columns={"index": "ticker"}),
    ),
    "capm_alpha_by_asset.png": (
        plot_factor_bar(capm_metrics, "alpha", "CAPM Alpha by Asset", "Annualized alpha"),
        "bar",
        capm_metrics[["alpha"]].rename(columns={"alpha": "value"}).reset_index().rename(columns={"index": "ticker"}),
    ),
    "benchmark_correlation_by_asset.png": (
        plot_factor_bar(capm_metrics, "correlation_to_benchmark", "Correlation to SPY by Asset", "Correlation"),
        "bar",
        capm_metrics[["correlation_to_benchmark"]].rename(columns={"correlation_to_benchmark": "value"}).reset_index().rename(columns={"index": "ticker"}),
    ),
    "rolling_beta_selected_assets.png": (
        plot_rolling_beta(rolling_beta, selected_assets=selected_rolling_assets),
        "lines",
        rolling_beta[selected_rolling_assets],
    ),
}

figure_export_methods = {}
for filename, (fig, chart_type, data) in figure_specs.items():
    figure_export_methods[filename] = save_plotly_or_pillow(fig, FIGURES_DIR / filename, chart_type, data)

figure_export_methods

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/_plotly_utils/basevalidators.py:105: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



{'capm_beta_by_asset.png': 'pillow_fallback',
 'capm_alpha_by_asset.png': 'pillow_fallback',
 'benchmark_correlation_by_asset.png': 'pillow_fallback',
 'rolling_beta_selected_assets.png': 'pillow_fallback'}

## Benchmark sensitivity interpretation

In [10]:
highest_beta = capm_metrics["beta"].idxmax()
lowest_beta = capm_metrics["beta"].idxmin()
highest_corr = capm_metrics["correlation_to_benchmark"].idxmax()
lowest_corr = capm_metrics["correlation_to_benchmark"].idxmin()

print(f"Highest beta asset: {highest_beta} ({capm_metrics.loc[highest_beta, 'beta']:.2f})")
print(f"Lowest beta asset: {lowest_beta} ({capm_metrics.loc[lowest_beta, 'beta']:.2f})")
print(f"Highest benchmark correlation: {highest_corr} ({capm_metrics.loc[highest_corr, 'correlation_to_benchmark']:.2f})")
print(f"Lowest benchmark correlation: {lowest_corr} ({capm_metrics.loc[lowest_corr, 'correlation_to_benchmark']:.2f})")

Highest beta asset: NVDA (1.82)
Lowest beta asset: JNJ (0.42)
Highest benchmark correlation: MSFT (0.79)
Lowest benchmark correlation: JNJ (0.43)


## Defensive vs aggressive asset discussion

- Assets with beta above 1.0 are more sensitive to SPY movements and can be interpreted as more aggressive within this single-index framework.
- Assets with beta below 1.0 are less sensitive to SPY and may behave more defensively relative to the broad equity benchmark.
- Correlation and tracking error matter alongside beta because an asset can have market sensitivity while still carrying meaningful idiosyncratic risk.
- Rolling beta helps show whether benchmark sensitivity is stable or changes across market regimes.

## Limitations

- SPY is a broad U.S. equity proxy, not a complete multi-factor benchmark.
- The risk-free rate is set to zero for this phase to keep the analysis simple and reproducible.
- Single-index regression explains only benchmark sensitivity; it does not isolate size, value, quality, momentum, or sector factors.
- The local environment may require the NumPy fallback regression path if statsmodels is unavailable or incompatible.
- Historical beta and alpha estimates are descriptive and not investment advice.

## Next steps for Phase 3 portfolio optimization

- Use the return and risk inputs from Phases 1 and 2 to define expected return and covariance assumptions.
- Build initial portfolio construction methods such as equal weight, minimum volatility, and maximum Sharpe portfolios.
- Keep optimization outputs interpretable before moving into backtesting.